In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load all datasets and concatenate them with a dataset name column
ours = {
    'Avila': 'ovr_results/ours/Avila_results.csv',
    'Chessgame': 'ovr_results/ours/Chessgame_results.csv',
    'Covertype': 'ovr_results/ours/Covertype_results.csv',
    'Dermatology': 'ovr_results/ours/Dermatology_results.csv',
    'HAR': 'ovr_results/ours/HAR_results.csv',
    'Land-use': 'ovr_results/ours/Land-use_results.csv',
    # 'Mfeat_icdm21': 'ovr_results/ours/Mfeat_icdm21_results.csv',
    'Mfeat': 'ovr_results/ours/Mfeat_results.csv',
    # 'Mosquitoes': 'ovr_results/ours/Mosquitoes_results.csv',
    'Nursery': 'ovr_results/ours/Nursery_results.csv',
    'PhishingURL': 'ovr_results/ours/PhishingURL_results.csv',
    'Satimage': 'ovr_results/ours/Satimage_results.csv',
    'Walking': 'ovr_results/ours/Walking_results.csv',
}

kaggle = {
    'Cirrhosis': 'ovr_results/cirrhosis_results.csv',
    'CustomerSegmentation': 'ovr_results/customer_segmentation_results.csv',
    # 'FashionMNIST': 'ovr_results/fashion-mnist_results.csv',
    # 'Healthcare': 'ovr_results/healthcare_results.csv',
    'IRIS': 'ovr_results/IRIS_results.csv',
    'MusicGenre': 'ovr_results/music_genre_results.csv',
    'PredictiveMaintenance': 'ovr_results/predictive_maintenance_results.csv',
    # 'StarClassification': 'ovr_results/star_classification_results.csv',
    'StudentPerformance': 'ovr_results/Student_performance_data_results.csv',
    'Zoo': 'ovr_results/zoo_results.csv',
}

openml = {
    'Spectrometer': 'ovr_results/dataset_313_spectrometer_results.csv',
    # 'AmazonReviews_1457': 'ovr_results/dataset_1457_amazon-commerce-reviews_results.csv',
    # 'OneHundredPlants_Margin': 'ovr_results/dataset_1491_one-hundred-plants-margin_results.csv',
    # 'OneHundredPlants_Shape': 'ovr_results/dataset_1492_one-hundred-plants-shape_results.csv',
    # 'OneHundredPlants_Texture': 'ovr_results/dataset_1493_one-hundred-plants-texture_results.csv',
    # 'BachChoralHarmony': 'ovr_results/dataset_4552_BachChoralHarmony_results.csv',
    #'AmazonReviews_seed0': 'ovr_results/dataset_44478_amazon-commerce-reviews_seed_0_nrows_2000_nclasses_10_ncols_100_stratify_True_results.csv',
   # 'AmazonReviews_seed1': 'ovr_results/dataset_44479_amazon-commerce-reviews_seed_1_nrows_2000_nclasses_10_ncols_100_stratify_True_results.csv',
   # 'AmazonReviews_seed2': 'ovr_results/dataset_44480_amazon-commerce-reviews_seed_2_nrows_2000_nclasses_10_ncols_100_stratify_True_results.csv',
   # 'AmazonReviews_seed3': 'ovr_results/dataset_44481_amazon-commerce-reviews_seed_3_nrows_2000_nclasses_10_ncols_100_stratify_True_results.csv',
    # 'AmazonReviews_seed4': 'ovr_results/dataset_44482_amazon-commerce-reviews_seed_4_nrows_2000_nclasses_10_ncols_100_stratify_True_results.csv',
    # 'FBI_Fingerprint': 'ovr_results/fabert_results.csv',
    # 'FARS': 'ovr_results/fars_results.csv',
    # 'Microaggregation2': 'ovr_results/microaggregation2_results.csv',
}

quapy = {
    'Academic-Success': 'ovr_results/quapy/academic-success_results.csv',
    'Digits': 'ovr_results/quapy/digits_results.csv',
    'Dry-Bean': 'ovr_results/quapy/dry-bean_results.csv',
    'Letter': 'ovr_results/quapy/letter_results.csv',
    'Wine-Quality': 'ovr_results/quapy/wine-quality_results.csv',
}


uci = {
    'Wine': 'ovr_results/uci/wine_results.csv',
}

# datasets = {**ours, **kaggle, **openml, **quapy, **uci}
datasets = {**kaggle, **openml}

# Load and concatenate all datasets
dfs = []
for dataset_name, filepath in datasets.items():
    temp_df = pd.read_csv(filepath)
    temp_df['dataset'] = dataset_name
    dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)
# Drop columns ending with '_p'
df = df.drop(columns=[col for col in df.columns if col.endswith('_p')])

In [ ]:
import re

pattern = r'^c(.+?)_(?:p_normalized|real)$'

classes = sorted(set([
    re.match(pattern, col).group(1) 
    for col in df.columns 
    if re.match(pattern, col)
]))
# classes

In [ ]:
import numpy as np

def normalized_cross_entropy(p, q, n_test=100):
    """
    p: true prevalence (array-like)
    q: estimated prevalence (array-like)
    n_test: test set size (int or array-like)

    returns: normCE(p, q)
    """
    eps = 0.5 / n_test

    p = np.clip(p, eps, 1 - eps)
    q = np.clip(q, eps, 1 - eps)

    ce_pq = -(p * np.log2(q) + (1 - p) * np.log2(1 - q))
    ce_pp = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

    return ce_pq - ce_pp

In [ ]:
# Calculate absolute error metrics
# Build all error columns at once to avoid fragmentation
error_cols = {}
for cls in classes:
    error_cols[f'c{cls}_error_normalized'] = abs(df[f'c{cls}_real'] - df[f'c{cls}_p_normalized'])

# Concatenate all new columns at once
df = pd.concat([df, pd.DataFrame(error_cols)], axis=1)

ce_cols = {}

for cls in classes:
    ce_cols[f'c{cls}_CEnorm'] = normalized_cross_entropy(
        p=df[f'c{cls}_real'].values,
        q=df[f'c{cls}_p_normalized'].values,
    )

# Concatenate all new columns at once
df = pd.concat([df, pd.DataFrame(ce_cols)], axis=1)

# Calculate MAE (Mean Absolute Error) for each row
# Sum all error columns and divide by number of non-NaN values
error_columns = [f'c{cls}_error_normalized' for cls in classes]
df['MAE'] = df[error_columns].mean(axis=1, skipna=True)

ce_columns = [f'c{cls}_CEnorm' for cls in classes]
df['mean_normCE'] = df[ce_columns].mean(axis=1, skipna=True)

In [ ]:
summarize_df = df[['qnt', 'MAE', 'mean_normCE', 'dataset']].copy()
# summarize_df['dataset'].unique()

# Analysis

## All results

In [ ]:
def plot_error_metrics_comparison(summarize_df):
    """
    Create an interactive box plot comparing error metrics (MAE and mean_normCE) 
    between traditional and synthetic methods.
    
    Parameters:
    summarize_df (pd.DataFrame): DataFrame containing qnt, MAE, mean_normCE, and dataset columns
    
    Returns:
    plotly.graph_objects.Figure: Interactive plotly figure
    """
    import plotly.graph_objects as go
    
    # Get unique datasets
    datasets_list = summarize_df['dataset'].unique().tolist()
    
    # Define metrics
    metrics = ['MAE', 'mean_normCE']
    
    # Define method pairs (traditional, synthetic)
    method_pairs = [
        ('ACC', 'ACC_syn'),
        ('PACC', 'PACC_syn'),
        ('X', 'X_syn'),
        ('MAX', 'MAX_syn'),
        ('T50', 'T50_syn'),
        ('MS', 'MS_syn'),
        ('MS2', 'MS2_syn'),
        ('SMM', 'SMM_syn'),
        ('HDy', 'HDy_syn'),
        ('DyS', 'DySyn')
    ]
    
    # Create figure
    fig = go.Figure()
    
    # Add traces for each dataset, metric, and method pair combination
    for dataset in datasets_list:
        for metric in metrics:
            df_dataset = summarize_df[summarize_df['dataset'] == dataset]
            
            for trad_method, syn_method in method_pairs:
                # Traditional method
                df_trad = df_dataset[df_dataset['qnt'] == trad_method]
                if len(df_trad) > 0:
                    fig.add_trace(go.Box(
                        y=df_trad[metric],
                        name=f'{trad_method}',
                        visible=(dataset == datasets_list[0] and metric == metrics[0]),
                        boxmean=True,
                        marker=dict(color='lightcoral'),
                        legendgroup=trad_method,
                        showlegend=True,
                        offsetgroup=trad_method,
                        width=0.4,
                        meta={'dataset': dataset, 'metric': metric, 'method': trad_method, 'type': 'trad'}
                    ))
                
                # Synthetic method
                df_syn = df_dataset[df_dataset['qnt'] == syn_method]
                if len(df_syn) > 0:
                    fig.add_trace(go.Box(
                        y=df_syn[metric],
                        name=f'{syn_method}',
                        visible=(dataset == datasets_list[0] and metric == metrics[0]),
                        boxmean=True,
                        marker=dict(color='lightblue'),
                        legendgroup=syn_method,
                        showlegend=True,
                        offsetgroup=syn_method,
                        width=0.4,
                        meta={'dataset': dataset, 'metric': metric, 'method': syn_method, 'type': 'syn'}
                    ))

            # Multiclass method
            # Add remaining quantifiers that don't have pairs
            remaining_methods = ['CC', 'PCC', 'PWK', 'HDx', 'GAC', 'GPAC', 'FM', 'EMQ', 'KDEyHD', 'KDEyCS', 'KDEyML']
            for method in remaining_methods:
                df_method = df_dataset[df_dataset['qnt'] == method]
                if len(df_method) > 0:
                    fig.add_trace(go.Box(
                        y=df_method[metric],
                        name=f'{method}',
                        visible=(dataset == datasets_list[0] and metric == metrics[0]),
                        boxmean=True,
                        marker=dict(color='lightgreen'),
                        legendgroup=method,
                        showlegend=True,
                        offsetgroup=method,
                        width=0.4,
                        meta={'dataset': dataset, 'metric': metric, 'method': method, 'type': 'other'}
                    ))

    
    # Create combined dropdown for Dataset - Metric
    combined_buttons = []
    for dataset in datasets_list:
        for metric in metrics:
            visible = []
            for trace in fig.data:
                visible.append(trace.meta['dataset'] == dataset and trace.meta['metric'] == metric)
            
            combined_buttons.append(dict(
                label=f'{dataset} - {metric}',
                method='update',
                args=[
                    {'visible': visible},
                    {'title': f'{metric} Distribution by Method - {dataset}',
                     'yaxis': {'title': metric}}
                ]
            ))
    
    # Update layout with combined dropdown
    fig.update_layout(
        updatemenus=[
            dict(
                active=0,
                buttons=combined_buttons,
                direction="down",
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.01,
                xanchor="left",
                y=1.15,
                yanchor="top"
            )
        ],
        annotations=[
            dict(text="Dataset - Metric:", showarrow=False,
                 x=0.01, y=1.18, xref="paper", yref="paper", align="right")
        ],
        title=f'{metrics[0]} Distribution by Method - {datasets_list[0]}',
        xaxis_title='Quantification Method',
        yaxis_title=metrics[0],
        height=600,
        showlegend=True,
        boxmode='group'
    )
    
    return fig

# Call the function and display the plot
fig_metrics = plot_error_metrics_comparison(summarize_df)
fig_metrics.show()

In [ ]:
# Create ranking of quantification methods by dataset based on MAE
ranking_by_dataset = summarize_df.groupby(['dataset', 'qnt'])['mean_normCE'].mean().reset_index()
ranking_by_dataset['rank'] = ranking_by_dataset.groupby('dataset')['mean_normCE'].rank(method='min')
ranking_by_dataset = ranking_by_dataset.sort_values(['dataset', 'rank'])

# Pivot table to show rankings in a more readable format
ranking_pivot = ranking_by_dataset.pivot(index='qnt', columns='dataset', values='rank')
ranking_pivot = ranking_pivot.fillna('-')

# print("Ranking of quantification methods by dataset (based on MAE):")
# print("Lower rank = better performance (lower MAE)")
# print("\n")
# print(ranking_pivot)

# Also create a summary showing average rank across all datasets
avg_ranking = ranking_by_dataset.groupby('qnt')['rank'].mean().reset_index()
avg_ranking.columns = ['qnt', 'avg_rank']
avg_ranking = avg_ranking.sort_values('avg_rank')

# print("\n\nAverage ranking across all datasets:")
# print(avg_ranking)

In [ ]:
import plotly.graph_objects as go

# Create ranking of quantification methods by dataset based on MAE
ranking_by_dataset_mae = summarize_df.groupby(['dataset', 'qnt'])['mean_normCE'].mean().reset_index()
ranking_by_dataset_mae['rank'] = ranking_by_dataset_mae.groupby('dataset')['mean_normCE'].rank(method='min')

# Create box plot of rankings per quantification method
fig = go.Figure()

# Get unique quantification methods sorted by median rank
qnt_methods = ranking_by_dataset_mae.groupby('qnt')['rank'].median().sort_values().index.tolist()

# Add box plot for each quantification method
for qnt in qnt_methods:
    qnt_data = ranking_by_dataset_mae[ranking_by_dataset_mae['qnt'] == qnt]
    fig.add_trace(go.Box(
        y=qnt_data['rank'],
        name=qnt,
        boxmean='sd',
        marker=dict(
            color='lightblue' if '_syn' in qnt or qnt in ['DySyn'] else 'lightcoral'
        )
    ))

# Update layout
fig.update_layout(
    title='Ranking Distribution of Quantification Methods Across Datasets (based on MAE)',
    xaxis_title='Quantification Method',
    yaxis_title='Rank (lower is better)',
    height=600,
    showlegend=False,
    # yaxis=dict(autorange='reversed')  # Lower rank at top
)

fig.show()

# Print summary statistics
# print("Summary statistics of rankings across datasets:")
# ranking_summary = ranking_by_dataset_mae.groupby('qnt')['rank'].agg(['mean', 'median', 'std', 'min', 'max']).round(2)
# ranking_summary = ranking_summary.sort_values('mean')
# print(ranking_summary)

In [ ]:
import plotly.graph_objects as go

# Create ranking of quantification methods by dataset based on MAE
ranking_by_dataset_mae = summarize_df.groupby(['dataset', 'qnt'])['mean_normCE'].mean().reset_index()
ranking_by_dataset_mae['rank'] = ranking_by_dataset_mae.groupby('dataset')['mean_normCE'].rank(method='min')

# Create box plot of rankings per quantification method
fig = go.Figure()

# Get unique quantification methods sorted by median rank
qnt_methods = ranking_by_dataset_mae.groupby('qnt')['rank'].median().sort_values().index.tolist()

# Add box plot for each quantification method
for qnt in qnt_methods:
    qnt_data = ranking_by_dataset_mae[ranking_by_dataset_mae['qnt'] == qnt]
    fig.add_trace(go.Box(
        y=qnt_data['rank'],
        name=qnt,
        boxmean='sd',
        marker=dict(
            color='lightblue' if '_syn' in qnt or qnt in ['DySyn'] else 'lightcoral'
        )
    ))

# Update layout
fig.update_layout(
    title='Ranking Distribution of Quantification Methods Across Datasets (based on mean_normCE)',
    xaxis_title='Quantification Method',
    yaxis_title='Rank (lower is better)',
    height=600,
    showlegend=False,
    # yaxis=dict(autorange='reversed')  # Lower rank at top
)

fig.show()

# Print summary statistics
# print("Summary statistics of rankings across datasets:")
# ranking_summary = ranking_by_dataset_mae.groupby('qnt')['rank'].agg(['mean', 'median', 'std', 'min', 'max']).round(2)
# ranking_summary = ranking_summary.sort_values('mean')
# print(ranking_summary)